In [149]:
import pandas as pd
import numpy as np
import re
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINEConv

In [150]:
matches = pd.read_csv("../data/processed/atp_matches.csv")
matches["tourney_date"] = pd.to_datetime(matches["tourney_date"])
matches = matches[matches["tourney_date"] < "2012-01-01"].copy()
matches["round_order"] = matches["round"].map(round_order_map)
matches = matches.sort_values(["tourney_date", "tourney_id", "round_order"]).reset_index(drop=True)
matches.head()
matches.tail()


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,round_order
7001,2011-352,Paris Masters,Hard,48.0,M,2011-11-07,3,104542,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,8.0,2935.0,1.0,13785.0,5
7002,2011-352,Paris Masters,Hard,48.0,M,2011-11-07,16,104545,NaN,NaN,...,31.0,17.0,13.0,1.0,3.0,25.0,1450.0,5.0,4390.0,5
7003,2011-352,Paris Masters,Hard,48.0,M,2011-11-07,36,103819,3.0,NaN,...,28.0,10.0,10.0,3.0,6.0,4.0,7530.0,7.0,3230.0,6
7004,2011-352,Paris Masters,Hard,48.0,M,2011-11-07,12,104542,6.0,NaN,...,61.0,19.0,17.0,7.0,7.0,8.0,2935.0,25.0,1450.0,6
7005,2011-352,Paris Masters,Hard,48.0,M,2011-11-07,13,103819,3.0,NaN,...,28.0,12.0,9.0,3.0,5.0,4.0,7530.0,8.0,2935.0,7


In [151]:
round_order_map = {
    "R128": 1,
    "R64": 2,
    "R32": 3,
    "R16": 4,
    "QF": 5,
    "SF": 6,
    "F": 7
}


In [152]:
snapshot_date = pd.Timestamp("2011-01-17")
graph_matches = matches[matches["tourney_date"] < snapshot_date].copy()

target_matches = matches[(matches["tourney_id"] == "2011-580") & (matches["round"] == "R128")].copy()
print(graph_matches.shape)
print(target_matches.shape)
target_matches[["tourney_id", "round", "winner_name", "loser_name", "surface"]].head()


(5800, 50)
(64, 50)


,tourney_id,round,winner_name,loser_name,surface
5800,2011-580,R128,Roger Federer,Lukas Lacko,Hard
5801,2011-580,R128,Xavier Malisse,Pablo Andujar,Hard
5802,2011-580,R128,Marcos Baghdatis,Grega Zemlja,Hard
5803,2011-580,R128,Juan Martin del Potro,Dudi Sela,Hard
5804,2011-580,R128,Pere Riba,Carsten Ball,Hard


In [153]:
def surface_one_hot(surface):
    if surface == "Hard":
        return [1, 0, 0]
    elif surface == "Clay":
        return [0, 1, 0]
    elif surface == "Grass":
        return [0, 0, 1]
    else:
        return [0, 0, 0]
    

In [154]:
def relative_game_diff(score_str):
    if pd.isna(score_str):
        return np.nan
    
    total_winner_games = 0
    total_loser_games = 0

    total_games = 0

    sets = str(score_str).split()

    for s in sets:
        s = re.sub(r"\(.*?\)", "", s)

        if re.match(r"^\d+-\d+$", s):
            a, b = s.split("-")
            total_winner_games += int(a)
            total_loser_games += int(b)

    total_games = total_winner_games + total_loser_games

    if total_games == 0:
        return np.nan
    else: 
        return ((total_winner_games -total_loser_games ) / total_games)


In [155]:
print(relative_game_diff("6-4 6-4"))
print(relative_game_diff("7-6(4) 3-6 6-3"))

graph_matches.head()

0.2
0.03225806451612903


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,round_order
0,2006-580,Australian Open,Hard,128.0,G,2006-01-16,52,103852,31.0,NaN,...,43.0,22.0,15.0,14.0,19.0,33.0,900.0,88.0,461.0,1
1,2006-580,Australian Open,Hard,128.0,G,2006-01-16,54,104468,NaN,Q,...,68.0,35.0,24.0,13.0,21.0,123.0,365.0,61.0,600.0,1
2,2006-580,Australian Open,Hard,128.0,G,2006-01-16,53,104607,19.0,NaN,...,49.0,23.0,19.0,3.0,9.0,23.0,1250.0,92.0,442.0,1
3,2006-580,Australian Open,Hard,128.0,G,2006-01-16,61,104198,NaN,NaN,...,40.0,20.0,16.0,2.0,6.0,110.0,393.0,31.0,920.0,1
4,2006-580,Australian Open,Hard,128.0,G,2006-01-16,62,103898,NaN,Q,...,21.0,5.0,9.0,4.0,10.0,162.0,255.0,95.0,429.0,1


In [156]:
players = pd.Index(
    pd.unique(
        pd.concat(
            [graph_matches["winner_name"], graph_matches["loser_name"]], 
            ignore_index=True
        )
    )
)

print(len(players))
players[:10]

player_to_idx = {players: i for i, players in enumerate(players)}
idx_to_player = {i: player for player, i in player_to_idx.items()}

list(player_to_idx.items())[:5]

407


[('Feliciano Lopez', 0),
 ('Gilles Simon', 1),
 ('Tomas Berdych', 2),
 ('Guillermo Garcia Lopez', 3),
 ('Julien Benneteau', 4)]

In [157]:
winner_static_matches = graph_matches[
    ["winner_name", "winner_ht", "winner_hand", "tourney_date"]
].rename(columns={"winner_name": "player",
                  "winner_ht": "height",
                  "winner_hand": "hand"})

loser_static_matches = graph_matches[
    ["loser_name", "loser_ht", "loser_hand", "tourney_date"]
].rename(columns={"loser_name": "player",
                  "loser_ht": "height",
                  "loser_hand": "hand"})

static_matches_df = pd.concat([winner_static_matches, loser_static_matches], ignore_index=True)
static_matches_df = (
    static_matches_df
    .sort_values("tourney_date")
    .groupby("player")
    .last()
)


static_matches_df.head()

,height,hand,tourney_date
player,,,
Adrian Mannarino,183.0,L,2010-08-30
Agustin Calleri,183.0,R,2009-06-22
Aisam Ul Haq Qureshi,183.0,R,2008-08-25
Alan Mackin,183.0,R,2007-01-15
Albert Costa,180.0,R,2006-04-24


In [158]:
df_gen = pd.read_csv("../data/processed/bscore_0_general.csv", index_col = 0)
bscore_gen_dict = df_gen.iloc[:, 0].to_dict()

df_clay = pd.read_csv("../data/processed/bscore_0_clay.csv", index_col = 0)
bscore_clay_dict = df_clay.iloc[:, 0].to_dict()

df_grass = pd.read_csv("../data/processed/bscore_0_grass.csv", index_col = 0)
bscore_grass_dict = df_grass.iloc[:, 0].to_dict()

df_hard = pd.read_csv("../data/processed/bscore_0_hard.csv", index_col = 0)
bscore_hard_dict = df_hard.iloc[:, 0].to_dict()

default_bscore = pd.Series(bscore_gen_dict).quantile(0.25)
default_bscore



np.float64(3.4789366760928304e-21)

In [159]:
def safe_score(d, player, default):
    value = d.get(player, default)
    if pd.isna(value):
        return float(default)
    return float(value)

median_height = static_matches_df["height"].dropna().median()
if pd.isna(median_height):
    median_height = 185.0

node_features = []

for player in players:
    bg = safe_score(bscore_gen_dict, player, default_bscore)
    bh = safe_score(bscore_hard_dict, player, default_bscore)
    bc = safe_score(bscore_clay_dict, player, default_bscore)
    bgra = safe_score(bscore_grass_dict, player, default_bscore)

    if player in static_matches_df.index:
        height = static_matches_df.loc[player, "height"]
        hand = static_matches_df.loc[player, "hand"]
    else:
        height = np.nan
        hand = np.nan

    if pd.isna(height):
        height = median_height

    height = float(height)

    is_left = 1.0 if hand == "L" else 0.0

    node_features.append([bg, bh, bc, bgra, height, is_left])

x = torch.tensor(node_features, dtype=torch.float)
x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

print(x.shape)
print("x nan:", torch.isnan(x).any().item())

torch.Size([407, 6])
x nan: False


In [160]:
alpha_days = 365
edge_rows = []

for _, row in graph_matches.iterrows():
    loser = row["loser_name"]
    winner = row["winner_name"]

    if loser not in player_to_idx or winner not in player_to_idx:
        continue

    src = player_to_idx[loser]
    dst = player_to_idx[winner]

    age_days = (snapshot_date -row["tourney_date"]).days
    recency_weight = 1 / (1 + age_days / alpha_days)

    rel_gd = relative_game_diff(row["score"])
    if pd.isna(rel_gd):
        rel_gd = 0.0
    surf = surface_one_hot(row["surface"])
    round_ord = float(row["round_order"]) if pd.notna(row["round_order"]) else 0.0

    edge_attr = [recency_weight, rel_gd] + surf + [round_ord]
    edge_rows.append([src, dst] + edge_attr)

In [161]:
edge_index = torch.tensor(
    [[row[0], row[1]] for row in edge_rows],
    dtype=torch.long
).t().contiguous()

edge_attr = torch.tensor(
    [row[2:] for row in edge_rows],
    dtype=torch.float
)

edge_attr = torch.nan_to_num(edge_attr, nan=0.0, posinf=0.0, neginf=0.0)

print("edge_attr nan:", torch.isnan(edge_attr).any().item())

edge_attr nan: False


In [162]:
bad_rows = []

for player in players:
    bg = bscore_gen_dict.get(player, default_bscore)
    bh = bscore_hard_dict.get(player, default_bscore)
    bc = bscore_clay_dict.get(player, default_bscore)
    bgra = bscore_grass_dict.get(player, default_bscore)

    if player in static_matches_df.index:
        height = static_matches_df.loc[player, "height"]
        hand = static_matches_df.loc[player, "hand"]
    else:
        height = np.nan
        hand = np.nan

    if pd.isna(height):
        height = median_height

    is_left = 1.0 if hand == "L" else 0.0

    vals = [bg, bh, bc, bgra, height, is_left]

    if any(pd.isna(v) for v in vals):
        bad_rows.append((player, vals, hand))

print("bad rows:", len(bad_rows))
print(bad_rows[:20])

bad rows: 0
[]


In [163]:
print(type(edge_rows))
print(len(edge_rows))
print(edge_rows[:3])

first = edge_rows[0]
print(type(first))
print(len(first))
print(first)

<class 'list'>
5800
[[303, 0, 0.16651459854014597, 0.3103448275862069, 1, 0, 0, 1.0], [83, 1, 0.16651459854014597, 0.061224489795918366, 1, 0, 0, 1.0], [129, 2, 0.16651459854014597, 0.21951219512195122, 1, 0, 0, 1.0]]
<class 'list'>
8
[303, 0, 0.16651459854014597, 0.3103448275862069, 1, 0, 0, 1.0]


In [164]:
edge_index = torch.tensor(
    [[row[0], row[1]] for row in edge_rows],
    dtype=torch.long
).t().contiguous()

edge_attr = torch.tensor(
    [row[2:] for row in edge_rows],
    dtype=torch.float
)

print(edge_index.shape)
print(edge_attr.shape)

torch.Size([2, 5800])
torch.Size([5800, 6])


In [165]:
data = Data(
    x=x,
    edge_index = edge_index,
    edge_attr = edge_attr
)

print(data)

Data(x=[407, 6], edge_index=[2, 5800], edge_attr=[5800, 6])


In [166]:
target_rows = []

for _, row in target_matches.iterrows():
    winner = row["winner_name"]
    loser = row["loser_name"]

    if winner not in player_to_idx or loser not in player_to_idx:
        continue

    winner_idx = player_to_idx[winner]
    loser_idx = player_to_idx[loser]

    match_surface = surface_one_hot(row["surface"])
    match_round = float(row["round_order"]) if pd.notna(row["round_order"]) else 0.0
    match_context = match_surface + [match_round]

    target_rows.append({
        "winner_idx": winner_idx,
        "loser_idx": loser_idx,
        "match_context": match_context,
        "winner_name": winner,
        "loser_name": loser
    })

print(len(target_rows))
print(target_rows[:3])

64
[{'winner_idx': 35, 'loser_idx': 262, 'match_context': [1, 0, 0, 1.0], 'winner_name': 'Roger Federer', 'loser_name': 'Lukas Lacko'}, {'winner_idx': 10, 'loser_idx': 173, 'match_context': [1, 0, 0, 1.0], 'winner_name': 'Xavier Malisse', 'loser_name': 'Pablo Andujar'}, {'winner_idx': 8, 'loser_idx': 289, 'match_context': [1, 0, 0, 1.0], 'winner_name': 'Marcos Baghdatis', 'loser_name': 'Grega Zemlja'}]


In [167]:
alpha_days = 365
edge_rows = []

for _, row in graph_matches.iterrows():
    loser = row["loser_name"]
    winner = row["winner_name"]

    if loser not in player_to_idx or winner not in player_to_idx:
        continue

    src = player_to_idx[loser]
    dst = player_to_idx[winner]

    age_days = (snapshot_date - row["tourney_date"]).days
    recency_weight = 1 / (1 + age_days / alpha_days)

    rel_gd = relative_game_diff(row["score"])
    if pd.isna(rel_gd):
        rel_gd = 0.0

    surf = surface_one_hot(row["surface"])
    round_ord = float(row["round_order"]) if pd.notna(row["round_order"]) else 0.0

    edge_attr = [recency_weight, rel_gd] + surf + [round_ord]
    edge_rows.append([src, dst] + edge_attr)


In [168]:
winner_idx_tensor = torch.tensor(
    [r["winner_idx"] for r in target_rows],
    dtype = torch.long
)

loser_idx_tensor = torch.tensor(
    [r["loser_idx"] for r in target_rows],
    dtype = torch.long
)

match_context_tensor = torch.tensor(
    [r["match_context"] for r in target_rows],
    dtype = torch.float
)

print(winner_idx_tensor.shape)
print(loser_idx_tensor.shape)
print(match_context_tensor.shape)

torch.Size([64])
torch.Size([64])
torch.Size([64, 4])


In [169]:
class TennisGINE(nn.Module):
    def __init__(self, node_in_dim, edge_in_dim, hidden_dim, match_context_dim):
        super().__init__()

        self.node_encoder = nn.Linear(node_in_dim, hidden_dim)

        nn1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        nn2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.conv1 = GINEConv(nn1, edge_dim = edge_in_dim)
        self.conv2 = GINEConv(nn2, edge_dim = edge_in_dim)

        self.match_head = nn.Sequential(
            nn.Linear(hidden_dim + match_context_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, data, player_a_idx, player_b_idx, match_context):
        h = self.node_encoder(data.x)
        h = self.conv1(h, data.edge_index, data.edge_attr)
        h = F.relu(h)
        h = self.conv2(h, data.edge_index, data.edge_attr)

        h_a = h[player_a_idx]
        h_b = h[player_b_idx]

        diff = h_a - h_b
        z = torch.cat([diff, match_context], dim = 1)

        logits = self.match_head(z).squeeze(-1)
        probs = torch.sigmoid(logits)

        return probs, logits

In [170]:
edge_index = torch.tensor(
    [[row[0], row[1]] for row in edge_rows],
    dtype=torch.long
).t().contiguous()

edge_attr = torch.tensor(
    [row[2:] for row in edge_rows],
    dtype=torch.float
)

edge_attr = torch.nan_to_num(edge_attr, nan=0.0, posinf=0.0, neginf=0.0)

print(type(edge_attr))
print(edge_index.shape)
print(edge_attr.shape)

data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr
)

print(data)

<class 'torch.Tensor'>
torch.Size([2, 5800])
torch.Size([5800, 6])
Data(x=[407, 6], edge_index=[2, 5800], edge_attr=[5800, 6])


In [171]:
model = TennisGINE(
    node_in_dim = x.shape[1],
    edge_in_dim = edge_attr.shape[1],
    hidden_dim = 32,
    match_context_dim = match_context_tensor.shape[1]
)

model

TennisGINE(
  (node_encoder): Linear(in_features=6, out_features=32, bias=True)
  (conv1): GINEConv(nn=Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
  ))
  (conv2): GINEConv(nn=Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
  ))
  (match_head): Sequential(
    (0): Linear(in_features=36, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [172]:
probs, logits = model(
    data,
    winner_idx_tensor,
    loser_idx_tensor,
    match_context_tensor
)

print(probs[:10])
print(logits[:10])
print(probs.shape, logits.shape)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.7983, 1.0000,
        1.0000], grad_fn=<SliceBackward0>)
tensor([3.8727e+03, 1.8100e+02, 8.4928e+02, 8.4538e+02, 2.8525e+01, 5.8297e+02,
        6.4072e+02, 1.3756e+00, 4.8268e+01, 3.7407e+01],
       grad_fn=<SliceBackward0>)
torch.Size([64]) torch.Size([64])


In [173]:
rng = np.random.default_rng(42)

supervised_rows = []

for _, row in target_matches.iterrows():
    winner = row["winner_name"]
    loser = row["loser_name"]

    if winner not in player_to_idx or loser not in player_to_idx:
        continue

    winner_idx = player_to_idx[winner]
    loser_idx = player_to_idx[loser]

    match_surface = surface_one_hot(row["surface"])
    match_round = float(row["round_order"]) if pd.notna(row["round_order"]) else 0.0
    match_context = match_surface + [match_round]

    if rng.random() < 0.5:
        player_a_idx = winner_idx
        player_b_idx = loser_idx
        y = 1.0
    else:
        player_a_idx = loser_idx
        player_b_idx = winner_idx
        y = 0.0

    supervised_rows.append({
        "player_a_idx": player_a_idx,
        "player_b_idx": player_b_idx,
        "match_context": match_context,
        "y": y,
        "winner_name": winner,
        "loser_name": loser
    })

print(len(supervised_rows))
supervised_rows[:3]

64


[{'player_a_idx': 262,
  'player_b_idx': 35,
  'match_context': [1, 0, 0, 1.0],
  'y': 0.0,
  'winner_name': 'Roger Federer',
  'loser_name': 'Lukas Lacko'},
 {'player_a_idx': 10,
  'player_b_idx': 173,
  'match_context': [1, 0, 0, 1.0],
  'y': 1.0,
  'winner_name': 'Xavier Malisse',
  'loser_name': 'Pablo Andujar'},
 {'player_a_idx': 289,
  'player_b_idx': 8,
  'match_context': [1, 0, 0, 1.0],
  'y': 0.0,
  'winner_name': 'Marcos Baghdatis',
  'loser_name': 'Grega Zemlja'}]

In [174]:
player_a_idx_tensor = torch.tensor(
    [r["player_a_idx"] for r in supervised_rows],
    dtype = torch.long
)

player_b_idx_tensor = torch.tensor(
    [r["player_b_idx"] for r in supervised_rows],
    dtype = torch.long
)

match_context_sup_tensor = torch.tensor(
    [r["match_context"] for r in supervised_rows],
    dtype = torch.float
)

y_tensor = torch.tensor(
    [r["y"] for r in supervised_rows],
    dtype = torch.float
)

print(player_a_idx_tensor.shape)
print(player_b_idx_tensor.shape)
print(match_context_sup_tensor.shape)
print(y_tensor.shape)

torch.Size([64])
torch.Size([64])
torch.Size([64, 4])
torch.Size([64])


In [175]:
probs, logits = model(
    data,
    player_a_idx_tensor,
    player_b_idx_tensor,
    match_context_sup_tensor
)

print(probs[:10])
print(logits[:10])
print(y_tensor[:10])
print(probs.shape, logits.shape)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.8155, 1.0000,
        1.0000], grad_fn=<SliceBackward0>)
tensor([926.3740, 181.0035, 202.7276, 201.7137,  28.5251, 138.3691, 152.9652,
          1.4861,  48.2685,  37.4070], grad_fn=<SliceBackward0>)
tensor([0., 1., 0., 0., 1., 0., 0., 0., 1., 1.])
torch.Size([64]) torch.Size([64])


In [176]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [177]:
for epoch in range(200):
    model.train()
    optimizer.zero_grad()

    probs, logits = model(
        data,
        player_a_idx_tensor,
        player_b_idx_tensor,
        match_context_sup_tensor
    )

    loss = criterion(logits, y_tensor)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            probs_eval, logits_eval = model(
                data,
                player_a_idx_tensor,
                player_b_idx_tensor,
                match_context_sup_tensor
            )
            preds = (probs_eval >= 0.5).float()
            acc = (preds == y_tensor).float().mean().item()

        print(f"epoch={epoch:03d} loss={loss.item():.4f} acc={acc:.4f}")

epoch=000 loss=93.1404 acc=0.6875
epoch=020 loss=3.9847 acc=0.6562
epoch=040 loss=3.8693 acc=0.7188
epoch=060 loss=8.5098 acc=0.7031
epoch=080 loss=3.0306 acc=0.7344
epoch=100 loss=2.5887 acc=0.7188
epoch=120 loss=3.1674 acc=0.7344
epoch=140 loss=2.0350 acc=0.7500
epoch=160 loss=1.4119 acc=0.7656
epoch=180 loss=0.4627 acc=0.5938


In [178]:
model.eval()

with torch.no_grad():
    probs, logits = model(
        data,
        player_a_idx_tensor,
        player_b_idx_tensor,
        match_context_sup_tensor
    )

preds = (probs >= 0.5).float()

for i in range(min(10, len(supervised_rows))):
    print(
        i,
        "p =", round(probs[i].item(), 4),
        "pred =", int(preds[i].item()),
        "true =", int(y_tensor[i].item())
    )

0 p = 0.0 pred = 0 true = 0
1 p = 0.9472 pred = 1 true = 1
2 p = 0.0 pred = 0 true = 0
3 p = 0.0 pred = 0 true = 0
4 p = 0.4602 pred = 0 true = 1
5 p = 0.0 pred = 0 true = 0
6 p = 0.0 pred = 0 true = 0
7 p = 0.519 pred = 1 true = 0
8 p = 0.6191 pred = 1 true = 1
9 p = 0.0085 pred = 0 true = 1


In [179]:
print("x nan:", torch.isnan(x).any().item())
print("x inf:", torch.isinf(x).any().item())

print("edge_attr nan:", torch.isnan(edge_attr).any().item())
print("edge_attr inf:", torch.isinf(edge_attr).any().item())

print("match_context nan:", torch.isnan(match_context_sup_tensor).any().item())
print("match_context inf:", torch.isinf(match_context_sup_tensor).any().item())

print("y nan:", torch.isnan(y_tensor).any().item())
print("y inf:", torch.isinf(y_tensor).any().item())

x nan: False
x inf: False
edge_attr nan: False
edge_attr inf: False
match_context nan: False
match_context inf: False
y nan: False
y inf: False


In [180]:
print(static_matches_df["height"].isna().sum())
print(static_matches_df["height"].median())

14
185.0


In [181]:
print("general size:", len(bscore_gen_dict))
print("hard size:", len(bscore_hard_dict))
print("clay size:", len(bscore_clay_dict))
print("grass size:", len(bscore_grass_dict))

print("general nan:", pd.Series(bscore_gen_dict).isna().any())
print("hard nan:", pd.Series(bscore_hard_dict).isna().any() if len(bscore_hard_dict) > 0 else "empty")
print("clay nan:", pd.Series(bscore_clay_dict).isna().any() if len(bscore_clay_dict) > 0 else "empty")
print("grass nan:", pd.Series(bscore_grass_dict).isna().any() if len(bscore_grass_dict) > 0 else "empty")

general size: 407
hard size: 318
clay size: 291
grass size: 292
general nan: False
hard nan: False
clay nan: False
grass nan: False
